In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

minsearch is a lightweight, in-memory text search library developed for the LLM Zoomcamp course as a simplified alternative to heavier search engines like Elasticsearch. It is designed for easy document indexing and retrieval in Python, allowing users to define fields as either text (which are searchable and scored using a BM25-like algorithm) or keyword (which are used for exact matching and filtering). Because it runs entirely in memory, it is exceptionally fast for prototyping Retrieval-Augmented Generation (RAG) pipelines and agentic systems, providing a straightforward way to turn a local collection of documents into a searchable knowledge base.

In [4]:
import minsearch

# 2. Index the documents with minsearch
# 'content' is the text field and 'filename' is the keyword field
index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

# 3. Search with the specific query
query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(
    query=query,
    num_results=5
)

# 4. Identify the filename of the first result
if results:
    print(f"The filename of the first result is: {results[0]['filename']}")
else:
    print("No results found.")


The filename of the first result is: 01-agentic-rag/lessons/14-agentic-loop.md


In [ ]:
# ! wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-07 11:51:07--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.1’

rag_helper.py.1     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-07 11:51:07 (44.5 MB/s) - ‘rag_helper.py.1’ saved [2134/2134]



tiktoken is a high-performance, open-source Byte Pair Encoding (BPE) tokenizer developed by OpenAI. It is specifically designed for use with OpenAI's models, such as GPT-4, GPT-4o, and GPT-3.5. tiktoken is significantly faster than many existing tokenization libraries, making it ideal for processing large volumes of text in real-time. For developers, its primary utility lies in accurately counting tokens to manage model context window limits and estimate API usage costs. Because tokenization happens locally, it does not require an active OpenAI API key or a network connection to operate.

In [5]:
import tiktoken
from dataclasses import dataclass
from typing import Dict, Any, List, Tuple

# 3. Modify the RAG structure to expose usage
@dataclass
class Usage:
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int

@dataclass
class RAGResponse:
    answer: str
    usage: Usage

class RAGAssistant:
    def __init__(self, index, model_name="gpt-4o"):
        self.index = index
        # We use tiktoken to simulate the tokenizer used by gpt-4o/mini models
        self.encoding = tiktoken.encoding_for_model("gpt-4o")

    def search(self, query: str) -> List[Dict[str, Any]]:
        # Fetching top 5 results as standard context
        return self.index.search(query=query, num_results=5)

    def build_context(self, search_results: List[Dict[str, Any]]) -> str:
        context = ""
        for doc in search_results:
            context += f"File: {doc['filename']}\nContent: {doc['content']}\n\n"
        return context

    def build_prompt(self, query: str, context: str) -> str:
        prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the course lessons.
Use only the facts from the CONTEXT. If the answer is not in the CONTEXT, say you don't know.

QUESTION: {question}

CONTEXT: 
{context}
""".strip()
        return prompt_template.format(question=query, context=context)

    def dummy_llm(self, prompt: str) -> RAGResponse:
        # Calculate tokens for the input prompt
        input_tokens = len(self.encoding.encode(prompt))
        
        # Simulate a typical response content
        answer = "The agentic loop stops when the model returns a final answer without any tool calls."
        output_tokens = len(self.encoding.encode(answer))
        
        usage = Usage(
            prompt_tokens=input_tokens,
            completion_tokens=output_tokens,
            total_tokens=input_tokens + output_tokens
        )
        
        return RAGResponse(answer=answer, usage=usage)

    def rag(self, query: str) -> Tuple[str, Usage]:
        results = self.search(query)
        context = self.build_context(results)
        prompt = self.build_prompt(query, context)
        
        response = self.dummy_llm(prompt)
        return response.answer, response.usage

# 4. Run the query and count tokens
query = "How does the agentic loop keep calling the model until it stops?"
assistant = RAGAssistant(index)
answer, usage = assistant.rag(query)

print(f"Query: {query}")
print(f"Number of input (prompt) tokens: {usage.prompt_tokens}")


Query: How does the agentic loop keep calling the model until it stops?
Number of input (prompt) tokens: 7122


In [6]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [7]:
# 4. Output the result
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 295


In [8]:
# 2. Setup Tokenizer (proxy for gpt-5.4-mini)
encoding = tiktoken.encoding_for_model("gpt-4o")

def count_tokens(text):
    return len(encoding.encode(text))

# 3. RAG Logic
prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the course lessons.
Use only the facts from the CONTEXT. If the answer is not in the CONTEXT, say you don't know.

QUESTION: {question}

CONTEXT: 
{context}
""".strip()

def build_prompt(query, search_results):
    context = ""
    for doc in search_results:
        context += f"File: {doc['filename']}\nContent: {doc['content']}\n\n"
    
    return prompt_template.format(question=query, context=context)

query = "How does the agentic loop keep calling the model until it stops?"

# --- Step A: Calculate Tokens for Full Documents (Q3) ---
index_full = minsearch.Index(text_fields=["content"], keyword_fields=["filename"])
index_full.fit(documents)
results_full = index_full.search(query=query, num_results=5)
prompt_full = build_prompt(query, results_full)
tokens_full = count_tokens(prompt_full)

# --- Step B: Calculate Tokens for Chunked Documents (Q5) ---
chunks = chunk_documents(documents, size=2000, step=1000)
index_chunk = minsearch.Index(text_fields=["content"], keyword_fields=["filename"])
index_chunk.fit(chunks)
results_chunk = index_chunk.search(query=query, num_results=5)
prompt_chunk = build_prompt(query, results_chunk)
tokens_chunk = count_tokens(prompt_chunk)

# --- Comparison ---
print(f"Tokens (Full Docs): {tokens_full}")
print(f"Tokens (Chunked): {tokens_chunk}")
print(f"Reduction ratio: {tokens_full / tokens_chunk:.1f}x")

Tokens (Full Docs): 7122
Tokens (Chunked): 2305
Reduction ratio: 3.1x


In [9]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# OpenRouter initialization
openai_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"), # Make sure this key is in your .env file
)

In [12]:
import json
import minsearch
from gitsource import GithubRepositoryDataReader, chunk_documents

# 1. Load and Chunk the data
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = [f.parse() for f in files]
chunks = chunk_documents(documents, size=2000, step=1000)

# 2. Index the chunks
index = minsearch.Index(
    text_fields=["content"], 
    keyword_fields=["filename"]
)
index.fit(chunks)

# 3. Define the Search Tool with a counter
search_call_count = 0

def search_tool_fn(query: str):
    """
    Search the course lessons for information using the given keywords.
    """
    global search_call_count
    search_call_count += 1
    return index.search(query=query, num_results=5)

# Tool schema for OpenAI-compatible APIs
tools = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search the course lessons for information",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                },
                "required": ["query"]
            }
        }
    }
]

# 4. Agentic Loop Implementation
def run_agent(question: str, client):
    instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
""".strip()

    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ]

    while True:
        # Note: Using gpt-4o as a proxy for gpt-5.4-mini logic
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
            # max_tokens=2000
        )
        
        message = response.choices[0].message
        messages.append(message)

        if not message.tool_calls:
            break

        for tool_call in message.tool_calls:
            if tool_call.function.name == "search":
                args = json.loads(tool_call.function.arguments)
                result = search_tool_fn(args['query'])
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                })

    return message.content

# 5. Execute and check results
# In a real environment, you would provide your OpenAI client here.
question = "How does the agentic loop work, and how is it different from plain RAG?"
answer = run_agent(question, openai_client)
print(f"Search was called {search_call_count} times.")


Search was called 2 times.
